In [1]:
# %% [markdown]
# # Task 3: Game Theory – Participants and Scenarios
# ## Asset: E-mini S&P 500 (ES)
# ## Goal: Identify market participants, their behavior, and build scenario tables

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from supabase import create_client
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
print("✅ Libraries loaded")

✅ Libraries loaded


In [2]:
# %% [markdown]
# ## 1. Load ES 15min data (full history)

# %%
def load_es_15min_with_progress(start_date=None, end_date=None, session_only=False):
    supabase = create_client(
        os.getenv("SUPABASE_URL"),
        os.getenv("SUPABASE_SERVICE_ROLE_KEY")
    )
    count_res = supabase.table('es_15min').select('*', count='exact').limit(0).execute()
    total = count_res.count
    print(f"📊 Total rows available: {total}")
    
    all_data = []
    offset = 0
    batch = 1000
    with tqdm(total=total, desc="Downloading data", unit="rows") as pbar:
        while True:
            query = supabase.table('es_15min').select('*').order('timestamp').range(offset, offset + batch - 1)
            if start_date:
                query = query.gte('timestamp', start_date)
            if end_date:
                query = query.lte('timestamp', end_date)
            response = query.execute()
            if not response.data:
                break
            all_data.extend(response.data)
            pbar.update(len(response.data))
            offset += batch
            if len(response.data) < batch:
                break
    
    df = pd.DataFrame(all_data)
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
    df = df.sort_values('timestamp').reset_index(drop=True)
    if session_only:
        df = df[(df['timestamp'].dt.hour >= 9) & (df['timestamp'].dt.hour < 16)]
        df = df[~((df['timestamp'].dt.hour == 9) & (df['timestamp'].dt.minute < 30))]
        df = df[~((df['timestamp'].dt.hour == 16) & (df['timestamp'].dt.minute > 0))]
    return df

print("Loading ES 15min data...")
df = load_es_15min_with_progress(session_only=False)
print(f"✅ Loaded {len(df):,} candles")
print(f"Period: {df['timestamp'].min()} → {df['timestamp'].max()}")

Loading ES 15min data...
📊 Total rows available: 144201


✅ Loaded 144,201 candles
Period: 2020-02-02 18:00:00 → 2026-05-07 00:45:00


In [3]:
# %% [markdown]
# ## 2. Identify main market participants in ES futures

# %%
# This is a conceptual cell – we will document based on industry knowledge and observed patterns
participants = {
    "Large Institutions / Banks": {
        "horizon": "Weeks to months",
        "behavior": "Provide liquidity, hedge portfolios, trade size. Often fade extreme moves.",
        "signals": "Extreme order flow, large open interest changes."
    },
    "Momentum Traders (CTAs)": {
        "horizon": "Days to weeks",
        "behavior": "Follow trends, enter after breakouts. Amplify moves.",
        "signals": "Breakout of 20-day highs/lows, strong directional moves."
    },
    "Value / Mean-Reversion Traders": {
        "horizon": "Intraday to a few days",
        "behavior": "Buy dips, sell rips. Use Bollinger Bands, RSI extremes.",
        "signals": "Overbought/oversold conditions, VWAP deviations."
    },
    "Retail / Day Traders": {
        "horizon": "Minutes to hours",
        "behavior": "Chase news, trade range breakouts. High emotional impact.",
        "signals": "High volume spikes, large intraday swings."
    }
}

participants_df = pd.DataFrame(participants).T
print("Main Market Participants:")
print(participants_df)

Main Market Participants:
                                               horizon  \
Large Institutions / Banks             Weeks to months   
Momentum Traders (CTAs)                  Days to weeks   
Value / Mean-Reversion Traders  Intraday to a few days   
Retail / Day Traders                  Minutes to hours   

                                                                         behavior  \
Large Institutions / Banks      Provide liquidity, hedge portfolios, trade siz...   
Momentum Traders (CTAs)         Follow trends, enter after breakouts. Amplify ...   
Value / Mean-Reversion Traders  Buy dips, sell rips. Use Bollinger Bands, RSI ...   
Retail / Day Traders            Chase news, trade range breakouts. High emotio...   

                                                                          signals  
Large Institutions / Banks       Extreme order flow, large open interest changes.  
Momentum Traders (CTAs)         Breakout of 20-day highs/lows, strong directio...  
Value

In [5]:
# %% [markdown]
# ## 3. Participant behavior under different market conditions

# %%
# Create a table of expected behavior per participant in three regimes: low volatility, high volatility, trending
conditions = ["Low Volatility", "High Volatility", "Strong Trend"]
behavior_table = pd.DataFrame(index=participants.keys(), columns=conditions)

behavior_table.loc["Large Institutions / Banks"] = [
    "Collect premiums; sell options; provide liquidity",
    "Hedge aggressively; may cause reversals",
    "Reinforce trend if aligned with fundamentals"
]
behavior_table.loc["Momentum Traders (CTAs)"] = [
    "Inactive; wait for breakout",
    "Enter on break of range; amplify moves",
    "Add to positions; trend followers"
]
behavior_table.loc["Value / Mean-Reversion Traders"] = [
    "Active; fading small moves",
    "Look for extremes to fade; high risk",
    "Fade pullbacks; counter-trend entries"
]
behavior_table.loc["Retail / Day Traders"] = [
    "Scalp small ranges; noise",
    "Panic during spikes; chase direction",
    "FOMO entries; late to trend"
]

print("Expected Behavior by Participant per Market Condition:")
print(behavior_table)

Expected Behavior by Participant per Market Condition:
                                                                   Low Volatility  \
Large Institutions / Banks      Collect premiums; sell options; provide liquidity   
Momentum Traders (CTAs)                               Inactive; wait for breakout   
Value / Mean-Reversion Traders                         Active; fading small moves   
Retail / Day Traders                                    Scalp small ranges; noise   

                                                        High Volatility  \
Large Institutions / Banks      Hedge aggressively; may cause reversals   
Momentum Traders (CTAs)          Enter on break of range; amplify moves   
Value / Mean-Reversion Traders     Look for extremes to fade; high risk   
Retail / Day Traders               Panic during spikes; chase direction   

                                                                Strong Trend  
Large Institutions / Banks      Reinforce trend if aligned with 

In [6]:
# %% [markdown]
# ## 4. Scenario Table (Key market situations)

# %%
scenarios = [
    {
        "Scenario": "Support/Resistance Test",
        "Description": "Price approaches a key historical level (e.g., all-time high or major low).",
        "Likely participants": "Institutions: defense. Momentum: wait for breakout. Retail: indecision.",
        "Potential outcome": "High volatility if level breaks; otherwise reversal."
    },
    {
        "Scenario": "After a Strong Move (3+% daily)",
        "Description": "Market had a sharp directional day; exhaustion possible.",
        "Likely participants": "Mean-reversion enters opposite. Momentum expects continuation.",
        "Potential outcome": "Often a pause or pullback of ~50% of the move."
    },
    {
        "Scenario": "Range Breakout (prolonged consolidation)",
        "Description": "Price compressed in a narrow range for many days.",
        "Likely participants": "Momentum triggers on breakout. Institutions may fade false breakouts.",
        "Potential outcome": "Explosive move in direction of breakout (often continuation)."
    }
]

scenarios_df = pd.DataFrame(scenarios)
print("Market Scenarios Table:")
print(scenarios_df.to_string())

Market Scenarios Table:
                                   Scenario                                                                  Description                                                      Likely participants                                              Potential outcome
0                   Support/Resistance Test  Price approaches a key historical level (e.g., all-time high or major low).  Institutions: defense. Momentum: wait for breakout. Retail: indecision.           High volatility if level breaks; otherwise reversal.
1           After a Strong Move (3+% daily)                     Market had a sharp directional day; exhaustion possible.           Mean-reversion enters opposite. Momentum expects continuation.                 Often a pause or pullback of ~50% of the move.
2  Range Breakout (prolonged consolidation)                            Price compressed in a narrow range for many days.    Momentum triggers on breakout. Institutions may fade false breakouts.  Explosive 

In [7]:
# %% [markdown]
# ## 5. (Optional) Visual example: Identify a range breakout period in data

# %%
# Compute daily range and look for periods with very low range (consolidation)
daily = df.resample('D', on='timestamp').agg({'high': 'max', 'low': 'min', 'close': 'last'}).dropna()
daily['range'] = daily['high'] - daily['low']
# Find lowest 10% range days
low_range_threshold = daily['range'].quantile(0.1)
consolidation_days = daily[daily['range'] <= low_range_threshold].index

print(f"Example consolidation days: {consolidation_days[:5].date}")
# You could then look at price action around those dates to illustrate breakout scenarios.

Example consolidation days: [datetime.date(2020, 2, 23) datetime.date(2020, 3, 4)
 datetime.date(2020, 3, 5) datetime.date(2020, 3, 6)
 datetime.date(2020, 6, 29)]


In [8]:
# %% [markdown]
# ## 6. Summary table of participants and their triggers

# %%
summary_participants = pd.DataFrame({
    "Participant": list(participants.keys()),
    "Typical Horizon": [participants[p]["horizon"] for p in participants],
    "Main Signal": [participants[p]["signals"] for p in participants]
})
print("Participant Summary with Typical Signals:")
print(summary_participants)

Participant Summary with Typical Signals:
                      Participant         Typical Horizon  \
0      Large Institutions / Banks         Weeks to months   
1         Momentum Traders (CTAs)           Days to weeks   
2  Value / Mean-Reversion Traders  Intraday to a few days   
3            Retail / Day Traders        Minutes to hours   

                                         Main Signal  
0   Extreme order flow, large open interest changes.  
1  Breakout of 20-day highs/lows, strong directio...  
2   Overbought/oversold conditions, VWAP deviations.  
3         High volume spikes, large intraday swings.  


## 📌 Conclusions – Task 3: Market Participants & Scenarios

### 1. Main Participants Identified
- **Large Institutions / Banks** (weeks to months): Provide liquidity, hedge. Signals: extreme order flow, large OI changes.
- **Momentum Traders (CTAs)** (days to weeks): Follow breakouts, amplify trends. Signals: 20-day highs/lows breakouts.
- **Mean-Reversion Traders** (intraday to few days): Buy dips, sell rips. Signals: overbought/oversold, VWAP deviations.
- **Retail / Day Traders** (minutes to hours): Chase news, range breakouts. Signals: volume spikes, intraday swings.

### 2. Expected Behavior by Market Condition

| Participant         | Low Volatility                 | High Volatility                     | Strong Trend                           |
|---------------------|--------------------------------|-------------------------------------|----------------------------------------|
| Institutions        | Sell options, provide liquidity| Hedge aggressively, may cause reversals | Reinforce if aligned with fundamentals |
| Momentum (CTAs)      | Inactive, wait for breakout    | Enter on breakouts, amplify moves    | Add to positions, trend followers      |
| Mean-Reversion      | Active, fade small moves       | Look for extremes to fade (risky)    | Fade pullbacks, counter-trend          |
| Retail              | Scalp small ranges             | Panic, chase direction               | FOMO entries, late to trend            |

### 3. Market Scenarios Constructed
1. **Support/Resistance Test**  
   - High volatility if broken; reversal if respected.
2. **After a Strong Move (>3% daily)**  
   - Often a pause or pullback of ~50%.
3. **Range Breakout (prolonged consolidation)**  
   - Explosive move in direction of breakout.

### 4. Implications for Trading Strategy
- At **support/resistance**: Act **contrarian** (fade) – institutions defend levels.
- **After a strong move**: Also favor **contrarian** (wait for pullback) or wait for confirmation.
- **Range breakout**: Best to **follow the majority** (momentum) – CTAs dominate.

These conclusions will feed into Task 4 (Payoff Matrices) and Task 5 (Trading Rules).